# Single SVS Learning With Slideflow Studio

This notebook is a small beginner workflow for one open-source `.svs` slide.

It keeps everything in one place:

1. load one slide
2. generate thumbnail and simple QC assets
3. estimate tissue regions
4. save a simple manifest and metadata summary
5. import Slideflow and prepare Studio view code
6. print exact Slideflow Studio launch commands


## Before You Run

- Update `SLIDE_PATH` to your `.svs` file.
- If you use the VM, activate `pathology310` first.
- This notebook does not modify the original slide.


In [ ]:
from pathlib import Path
import csv
import json
import shutil

import numpy as np
from openslide import OpenSlide
from PIL import Image, ImageEnhance


In [ ]:
# Update these paths for your own case.
SLIDE_PATH = Path(r"/path/to/your_slide.svs")
OUTPUT_DIR = Path("./outputs/demo_slide")
THUMB_WIDTH = 1600

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SLIDE_PATH

In [ ]:
def tissue_mask_from_thumbnail(image: Image.Image) -> np.ndarray:
    rgb = np.asarray(image.convert("RGB")).astype(np.uint8)
    bright = rgb.mean(axis=2)
    channel_spread = rgb.max(axis=2) - rgb.min(axis=2)
    return (bright < 225) & (channel_spread > 12)


def save_mask(mask: np.ndarray, path: Path) -> None:
    Image.fromarray((mask.astype(np.uint8) * 255), mode="L").save(path)


def save_overlay(base: Image.Image, mask: np.ndarray, path: Path) -> None:
    rgb = np.asarray(base.convert("RGB")).copy()
    rgb[mask] = (0.60 * rgb[mask] + 0.40 * np.array([40, 180, 255])).astype(np.uint8)
    Image.fromarray(rgb).save(path)


def write_manifest(slide_path: Path, output_dir: Path) -> Path:
    manifest_path = output_dir / "single_slide_manifest.csv"
    with manifest_path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(
            handle,
            fieldnames=["slide", "patient", "label", "source", "notes"],
        )
        writer.writeheader()
        writer.writerow(
            {
                "slide": slide_path.name,
                "patient": slide_path.stem.split(".")[0],
                "label": "unknown",
                "source": "open_source_demo",
                "notes": "Single-slide learning manifest for Slideflow exploration.",
            }
        )
    return manifest_path


In [ ]:
slide = OpenSlide(str(SLIDE_PATH))
width, height = slide.dimensions
aspect = height / max(width, 1)
thumb_height = max(1, int(THUMB_WIDTH * aspect))
thumbnail = slide.get_thumbnail((THUMB_WIDTH, thumb_height)).convert("RGB")

enhanced = ImageEnhance.Color(thumbnail).enhance(1.25)
enhanced = ImageEnhance.Contrast(enhanced).enhance(1.12)
enhanced = ImageEnhance.Sharpness(enhanced).enhance(1.08)

mask = tissue_mask_from_thumbnail(thumbnail)

thumbnail_path = OUTPUT_DIR / "thumbnail.png"
enhanced_path = OUTPUT_DIR / "enhanced_preview.png"
mask_path = OUTPUT_DIR / "tissue_mask.png"
overlay_path = OUTPUT_DIR / "tissue_overlay.png"
summary_path = OUTPUT_DIR / "slide_summary.json"

thumbnail.save(thumbnail_path)
enhanced.save(enhanced_path)
save_mask(mask, mask_path)
save_overlay(thumbnail, mask, overlay_path)
manifest_path = write_manifest(SLIDE_PATH, OUTPUT_DIR)

summary = {
    "slide_name": SLIDE_PATH.name,
    "slide_path": str(SLIDE_PATH.resolve()),
    "width": width,
    "height": height,
    "level_count": slide.level_count,
    "level_dimensions": [list(level) for level in slide.level_dimensions],
    "thumbnail_width": thumbnail.size[0],
    "thumbnail_height": thumbnail.size[1],
    "tissue_fraction_estimate": round(float(mask.mean()), 6),
    "selected_properties": {
        "openslide.vendor": slide.properties.get("openslide.vendor"),
        "openslide.objective-power": slide.properties.get("openslide.objective-power"),
        "openslide.mpp-x": slide.properties.get("openslide.mpp-x"),
        "openslide.mpp-y": slide.properties.get("openslide.mpp-y"),
        "aperio.AppMag": slide.properties.get("aperio.AppMag"),
    },
}
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")

summary

In [ ]:
from IPython.display import display

display(thumbnail)
display(enhanced)


In [ ]:
report = {
    "slide_name": SLIDE_PATH.name,
    "dimensions": list(slide.dimensions),
    "level_count": slide.level_count,
    "manifest": str(manifest_path.resolve()),
    "thumbnail": str(thumbnail_path.resolve()),
    "enhanced_preview": str(enhanced_path.resolve()),
    "tissue_mask": str(mask_path.resolve()),
    "tissue_overlay": str(overlay_path.resolve()),
    "slide_summary": str(summary_path.resolve()),
}

try:
    import slideflow as sf
    report["slideflow_installed"] = True
    report["slideflow_version"] = getattr(sf, "__version__", "unknown")
except Exception as exc:
    report["slideflow_installed"] = False
    report["slideflow_error"] = str(exc)

report


## Slideflow Import And Studio View Code

This cell shows the direct Slideflow import path you can use before opening Studio.

If GUI display is available, `wsi.view()` can open the interactive viewer path from Python.


In [ ]:
slideflow_view_info = {}

try:
    import slideflow as sf
    
    wsi = sf.WSI(str(SLIDE_PATH), tile_px=256, tile_um=128)
    slideflow_view_info = {
        "slideflow_imported": True,
        "slideflow_version": getattr(sf, "__version__", "unknown"),
        "wsi_repr": repr(wsi),
        "python_view_code": [
            "import slideflow as sf",
            f"wsi = sf.WSI(r'{str(SLIDE_PATH.resolve())}', tile_px=256, tile_um=128)",
            "wsi.view()",
        ],
        "note": "Run wsi.view() only when GUI display is available.",
    }
except Exception as exc:
    slideflow_view_info = {
        "slideflow_imported": False,
        "error": str(exc),
        "python_view_code": [
            "import slideflow as sf",
            f"wsi = sf.WSI(r'{str(SLIDE_PATH.resolve())}', tile_px=256, tile_um=128)",
            "wsi.view()",
        ],
    }

slideflow_view_info


In [ ]:
studio_cmd = shutil.which("slideflow-studio") or "slideflow-studio"
python_cmd = shutil.which("python") or "python"

studio_info = {
    "local_studio_command": studio_cmd,
    "local_module_command": f"{python_cmd} -m slideflow.studio",
    "vm_setup": [
        "source /opt/miniforge3/etc/profile.d/conda.sh",
        "conda activate /opt/miniforge3/envs/pathology310",
        "slideflow-studio",
    ],
    "open_this_slide_in_studio": str(SLIDE_PATH.resolve()),
    "python_view_code": [
        "import slideflow as sf",
        f"wsi = sf.WSI(r'{str(SLIDE_PATH.resolve())}', tile_px=256, tile_um=128)",
        "wsi.view()",
    ],
    "keep_this_manifest_beside_you": str(manifest_path.resolve()),
}

studio_info


## Studio Learning Flow

After running the cells above:

1. run the Slideflow import cell and review the generated `python_view_code`
2. open Slideflow Studio or run `wsi.view()` when GUI display is available
3. load the original `.svs` slide
4. compare it with `thumbnail.png` and `enhanced_preview.png`
5. use `tissue_overlay.png` to understand tissue-rich areas
6. use `slide_summary.json` and the one-row manifest for context
